# 02. Task Distribution

A team is only useful if work flows across it. Notebook `01-team-formation` got you a roster of `Entity` records and a working audit chain. That gives every team member a name, a role, and a verifiable identity — but it doesn't move tasks around. *That* is what this notebook is about.

`arcteam`'s task-distribution surface is intentionally small: a typed `Message` envelope, three URI schemes (`agent://`, `role://`, `channel://`), and a single `MessagingService` that routes envelopes onto append-only streams. The semantics are pull-based — receivers move a cursor forward, the sender never blocks waiting for delivery, and every routed message lands in the audit chain. From those primitives you compose the patterns you actually want: direct task assignment, role fan-out, project-scoped channels, priority queues, and dead-letter retries.

**You will learn:**

- Why task distribution belongs in `arcteam` and not on the agent (`arcagent`) or the runtime (`arcrun`).
- The full surface of `Channel`, `MsgType`, and `Priority` — what each value means and when to reach for it.
- How a task-assignment `Message` is shaped (typed body, refs, priority, action_required).
- How `MessagingService.send` routes a single envelope to one or many `to:` URIs.
- The pull-based receive loop: `poll`, `ack`, cursor monotonicity.
- How priorities interact with FIFO streams (Arc's deliberate, simple design).
- Failure and retry: DLQ entries, the four rejection reasons, audit residue.
- Where this stops and notebook `03` (channels in depth) and `04` (durability) pick up.

Notebook `01-team-formation` covers `TeamConfig`, `Entity`, and `EntityRegistry`; we build on that here without re-deriving it. DID basics live in `arctrust/01-identity-did` — we use `AgentIdentity` inline to bind sender/receiver identities to their registry handles, but we do not re-derive the DID model.

## 1. Setup

Standard Arc walkthrough preamble: locate the repo root, add every package's `src/` (and `tests/` where present) to `sys.path`, load `.env` if present. Identical across every Arc notebook — skim past once you've seen it.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Smoke-import the public surface this notebook touches. Every subsequent cell assumes these resolve cleanly.

In [ ]:
import arcteam
from arcteam import (
    AuditLogger,
    Channel,
    Cursor,
    Entity,
    EntityRegistry,
    EntityType,
    MemoryBackend,
    Message,
    MessagingService,
    MsgType,
    Priority,
)
from arctrust import AgentIdentity

print(f"arcteam version: {arcteam.__version__}")
print(
    "Loaded:",
    [
        Channel.__name__,
        Cursor.__name__,
        Message.__name__,
        MessagingService.__name__,
        MsgType.__name__,
        Priority.__name__,
        AgentIdentity.__name__,
    ],
)

## 2. Why distributed task assignment

A solo agent's `arcrun` loop already handles task execution — read tools, plan, dispatch, observe, repeat. The moment two agents share a goal, that loop stops being enough. *Who decides what each agent does? How does a planner hand a chunk of work to a researcher? How does a researcher signal back a result without polling the planner directly?*

Three different layers could in principle answer this, and the wrong split is how you get tangled, hard-to-audit systems:

| Layer | Tempting wrong answer | Correct boundary |
|---|---|---|
| `arcllm` | "Have the planner LLM mention the researcher in its output." | LLM is provider-agnostic call surface — knows nothing about teams. |
| `arcrun` | "Make the loop spawn child loops for sub-tasks." | Loop runs *one* agent's reason-act cycle. Doesn't know about other agents. |
| `arcagent` | "Add a `delegate_to` tool to the agent nucleus." | Agent owns its own tools, identity, and policy — not org structure. |
| `arcteam` | (right answer) | Multi-agent coordination, signed comms, roster-based addressing. |

From `packages/arcteam/CLAUDE.md`:

> **arcteam orchestrates multiple ArcAgent instances — it does not replace or duplicate agent internals.** Task distribution: assign, delegate, track work across agents. NOT execute tasks (that's arcrun).

The shape of the contract is: a sender constructs a `Message`, hands it to `MessagingService.send`, and the service routes it to one or more append-only streams (per-agent inbox, per-role fan-out, per-channel topic). Receivers `poll` their streams, process messages, and `ack` to advance their cursor. Senders never block on receivers; receivers never lose messages because cursors are persisted. That's the whole loop.

**The streams behind the scenes.** Every `to:` URI maps to a stream name on the storage backend:

| URI scheme | Stream name | Use |
|---|---|---|
| `agent://name` | `arc.agent.name` | Direct message / per-agent inbox |
| `user://name` | `arc.agent.name` | Same inbox space — users and agents share the address space |
| `role://name` | `arc.role.name` | Fan-out to anyone with that role |
| `channel://name` | `arc.channel.name` | Project- or topic-scoped multi-party channel |

All four are append-only. Each append gets a monotonic `seq`. That sequence is what makes the audit chain (and the cursor model) work.

In [ ]:
# Quick sanity check on the URI -> stream mapping the service uses internally.
from arcteam.messenger import _stream_name_from_uri

for uri in (
    "agent://planner",
    "user://josh",
    "role://workers",
    "channel://project-alpha",
):
    print(f"{uri:<28} -> {_stream_name_from_uri(uri)}")

## 3. `Channel`, `MsgType`, and `Priority` — the typed surface

Every `Message` carries two enums (`msg_type`, `priority`) and is routed against zero or more `Channel` records. These are tiny but load-bearing — a misclassified message is a missed deadline or an audit headache. We walk each one before sending anything.

### 3.1 `MsgType` — what *kind* of message this is

`MsgType` is a `StrEnum` with six values. It tags the *intent* of a message so receivers can filter, route, or escalate without parsing the body. Convention, not enforcement: a receiver decides what to do with each type.

| Value | Meaning |
|---|---|
| `INFO` | Pure broadcast / FYI. No action expected. |
| `REQUEST` | Asking for input. Reply expected. |
| `TASK` | Work assignment. Receiver is expected to execute and report back. |
| `RESULT` | Reply to a `REQUEST` or `TASK`. Body usually contains output. |
| `ALERT` | Something needs attention now. Often paired with `Priority.HIGH/CRITICAL`. |
| `ACK` | Lightweight acknowledgement that the receiver got the message. |

In [ ]:
for value in MsgType:
    print(f"  {value.name:<8}  ->  {value.value!r}")

# StrEnum: members compare equal to their string representation.
assert MsgType.TASK == "task"
assert MsgType.RESULT == "result"
print("\nStrEnum equality holds:", MsgType.TASK == "task")

### 3.2 `Priority` — *how urgent* this message is

Four levels. Like `MsgType`, this is metadata: the storage backend does not re-order messages by priority (streams are FIFO by `seq`). It's the *receiver's* job to consult `priority` and choose how to drain its inbox — covered in §7.

| Value | When to use |
|---|---|
| `LOW` | Background hints, eventual-consistency reports. |
| `NORMAL` | Default. Most task assignments and results. |
| `HIGH` | Time-sensitive — receiver should drain ahead of `NORMAL`. |
| `CRITICAL` | Reserved for incident-class alerts. Federal-tier deployments may pair with paging. |

In [ ]:
for value in Priority:
    print(f"  {value.name:<8}  ->  {value.value!r}")

# StrEnum members order alphabetically by default — define an explicit rank if you
# want a comparator. We'll use this in the priority section (§7).
PRIORITY_RANK = {
    Priority.CRITICAL: 0,
    Priority.HIGH: 1,
    Priority.NORMAL: 2,
    Priority.LOW: 3,
}
print("\nDescending urgency:")
for p, r in sorted(PRIORITY_RANK.items(), key=lambda kv: kv[1]):
    print(f"  rank={r}  {p.value}")

### 3.3 `Channel` — the multi-party topic

A `Channel` is a named, member-gated stream. Anyone in `members` can send to `channel://name` and read from it; non-members are rejected at send time with a DLQ entry (FR-7).

| Field | Type | Meaning |
|---|---|---|
| `name` | `str` | Unique per team. Maps to stream `arc.channel.{name}`. |
| `description` | `str` | Free-form. Audit-friendly. |
| `members` | `list[str]` | Allowed sender/receiver URIs (`agent://...`, `user://...`). |
| `created` | `str` | ISO-8601 timestamp; auto-set on `create_channel` if blank. |

Channels are the right call when more than two parties need a shared, durable conversation about the same topic — "project alpha standup", "incident-2026-05-08", "all-hands". For one-to-one task hand-off, prefer DM (`agent://`); for fan-out by capability, prefer role (`role://`).

In [ ]:
# A bare Channel record before it's persisted by MessagingService.
preview = Channel(
    name="project-alpha",
    description="Project Alpha - all hands",
    members=["agent://planner", "agent://researcher", "user://josh"],
)
for field, value in preview.model_dump().items():
    print(f"  {field:<13} {value!r}")

## 4. Crafting a task-assignment `Message`

`Message` is a 13-field Pydantic envelope. It maps cleanly to NATS JetStream once the team graduates beyond `MemoryBackend`, but the in-process service uses the same shape — that's the point.

| Field | Auto / Manual | Meaning |
|---|---|---|
| `seq` | auto (assigned at send) | Monotonic per-stream sequence number |
| `id` | auto | `msg_{ts_ns}_{counter}` — unique per process |
| `ts` | auto | UTC ISO-8601 timestamp |
| `sender` | required | URI of who sent it (must be registered) |
| `to` | required | List of one or more URIs to route to |
| `thread_id` | auto if absent | Self-id on first send; explicit on replies |
| `msg_type` | optional | One of `MsgType`; default `INFO` |
| `priority` | optional | One of `Priority`; default `NORMAL` |
| `action_required` | optional | Hint to receiver that they must act |
| `body` | required | UTF-8 string payload, ≤ 64 KB |
| `refs` | optional | Free-form list of cross-references (task IDs, file paths) |
| `status` | auto | Always `"sent"` after `send()` |
| `meta` | optional | Arbitrary dict (DLQ uses this; you can too) |

Best practice for tasks: encode a small structured payload as JSON in `body`, put a stable task ID in `refs`, and set `msg_type=MsgType.TASK` plus `action_required=True`.

In [ ]:
import json

# A typical task-assignment envelope before send: seq/id/ts are 0/empty/empty
# until MessagingService stamps them.
task_payload = {
    "task_id": "T-001",
    "title": "Summarize FY26 Q1 incident report",
    "deadline": "2026-05-09T17:00:00Z",
    "acceptance": ["3-bullet exec summary", "link to source"],
}

task_msg = Message(
    sender="agent://planner",
    to=["agent://researcher"],
    msg_type=MsgType.TASK,
    priority=Priority.HIGH,
    action_required=True,
    body=json.dumps(task_payload),
    refs=["task://T-001", "file://incidents/2026-q1.pdf"],
)
for k, v in task_msg.model_dump().items():
    print(f"  {k:<16} {v!r}")

**Body size is enforced at construction.** The 64 KB cap (`MAX_BODY_BYTES`) protects audit volume and keeps individual messages manageable in NATS. Pydantic validates this *before* the message ever reaches the service:

In [ ]:
from pydantic import ValidationError

# this raises — 70_000 bytes exceeds the 64 KB body cap
try:
    Message(sender="agent://planner", to=["agent://researcher"], body="x" * 70_000)
except ValidationError as e:
    print("ValidationError caught:")
    print("  ", str(e).splitlines()[0])
    print("  ", str(e).splitlines()[1])

**`thread_id` is the conversation key.** Leave it `None` on the first message in a conversation — the service will set it equal to the new `id`, so the message *is* its own thread root. On replies, set `thread_id=original.id` explicitly. Notebook `03` walks threading in depth; here it's enough to know that a thread's audit timeline is one consistent identifier.

## 5. Sending via `MessagingService`

Now the moving parts come together. `MessagingService` owns the actual routing logic. To stand one up you need three things:

1. A `StorageBackend` — `MemoryBackend` for in-process tests; `FileBackend` for durability (notebook `04`).
2. An `AuditLogger` — chains every operation. Notebook `01-team-formation` introduced this.
3. An `EntityRegistry` — sender validation depends on it; the service rejects unregistered senders.

We bootstrap a small team — planner, researcher, writer, plus a human operator — and bind each agent's registry handle to a real DID via `arctrust.AgentIdentity`. Then we send our first task.

In [ ]:
# Backend, audit, registry — same pattern as notebook 01.
backend = MemoryBackend()
audit = AuditLogger(backend, hmac_key=b"task-distribution-demo-key")
await audit.initialize()
registry = EntityRegistry(backend, audit)

# Inline DIDs: each agent gets a real Ed25519-backed identity.
planner_id = AgentIdentity.generate(org="acme", agent_type="planner")
researcher_id = AgentIdentity.generate(org="acme", agent_type="researcher")
writer_id = AgentIdentity.generate(org="acme", agent_type="writer")
operator_id = AgentIdentity.generate(org="acme", agent_type="operator")

print("DIDs:")
print("  planner:    ", planner_id.did)
print("  researcher: ", researcher_id.did)
print("  writer:     ", writer_id.did)
print("  operator:   ", operator_id.did)

Register each member with their DID stored as a capability tag (the same convention as notebook `01`).

In [ ]:
await registry.register(
    Entity(
        id="agent://planner",
        name="Planner",
        type=EntityType.AGENT,
        roles=["coordinator"],
        capabilities=["task.assign", f"did:{planner_id.did}"],
    )
)
await registry.register(
    Entity(
        id="agent://researcher",
        name="Researcher",
        type=EntityType.AGENT,
        roles=["worker", "research"],
        capabilities=["web.search", "doc.summarize", f"did:{researcher_id.did}"],
    )
)
await registry.register(
    Entity(
        id="agent://writer",
        name="Writer",
        type=EntityType.AGENT,
        roles=["worker", "writing"],
        capabilities=["doc.draft", f"did:{writer_id.did}"],
    )
)
await registry.register(
    Entity(
        id="user://josh",
        name="Josh",
        type=EntityType.USER,
        roles=["operator"],
        capabilities=[f"did:{operator_id.did}"],
    )
)

roster = await registry.list_entities()
print(f"Registered {len(roster)} members:")
for e in roster:
    print(f"  {e.id:<22}  roles={e.roles}")

Construct the `MessagingService`. It takes the backend, registry, and audit logger as collaborators — no hidden state. The `ui_reporter` argument is an optional duck-typed hook for live-observability pipelines (covered in `arcui` notebooks); we leave it `None` here.

In [ ]:
svc = MessagingService(backend, registry, audit)
print("MessagingService ready:", type(svc).__name__)

Create the `project-alpha` channel and add all four members. `create_channel` writes a `Channel` record and emits a `channel.created` audit event.

In [ ]:
await svc.create_channel(
    Channel(
        name="project-alpha",
        description="Project Alpha - planning and execution",
        members=[
            "agent://planner",
            "agent://researcher",
            "agent://writer",
            "user://josh",
        ],
    )
)
channels = await svc.list_channels()
for ch in channels:
    print(f"channel://{ch.name}  members={ch.members}")

### Three task-distribution patterns

With a roster and a channel in place, the same `Message` shape supports three different distribution patterns. The only thing that changes is the `to:` URI.

**Pattern 1 — Direct task assignment (DM).** Most common pattern for handing one piece of work to one specific agent. The receiver's inbox stream (`arc.agent.researcher`) gets exactly one new message.

In [ ]:
dm_task = Message(
    sender="agent://planner",
    to=["agent://researcher"],
    msg_type=MsgType.TASK,
    priority=Priority.NORMAL,
    action_required=True,
    body=json.dumps({"task_id": "T-001", "title": "Find prior incidents"}),
    refs=["task://T-001"],
)
sent = await svc.send(dm_task)
print(f"sent: id={sent.id}  seq={sent.seq}  thread_id={sent.thread_id}")
print(f"      ts={sent.ts}  status={sent.status}")

**Pattern 2 — Role fan-out.** When you don't care *which* worker picks it up, only *what kind*. Anyone with `role://worker` can poll `arc.role.worker` and grab the task. This is the classic work-queue pattern; in production you'd add an at-most-once locking convention on top, since a `role://` stream is fan-*out* not fan-*to-one*.

In [ ]:
role_task = Message(
    sender="agent://planner",
    to=["role://worker"],
    msg_type=MsgType.TASK,
    priority=Priority.NORMAL,
    body=json.dumps({"task_id": "T-002", "title": "Tag last week's logs"}),
    refs=["task://T-002"],
)
sent = await svc.send(role_task)
print(f"role fan-out -> arc.role.worker, seq={sent.seq}")

# Researcher polls the role stream and sees the work item.
msgs = await svc.poll("arc.role.worker", "agent://researcher")
for m in msgs:
    print(f"  researcher saw: seq={m.seq} type={m.msg_type.value} body={m.body!r}")

# Writer also has the worker role - they see the same item until somebody acks.
msgs = await svc.poll("arc.role.worker", "agent://writer")
for m in msgs:
    print(f"  writer saw:     seq={m.seq} body={m.body!r}")

**Pattern 3 — Channel broadcast.** Multi-party announcement. Everyone with channel membership reads the same stream. Useful for status updates, kickoffs, and "FYI" messages where you don't want one private conversation per recipient.

In [ ]:
broadcast = Message(
    sender="user://josh",
    to=["channel://project-alpha"],
    msg_type=MsgType.INFO,
    priority=Priority.NORMAL,
    body="Kickoff at 10:00 UTC. Async OK.",
)
sent = await svc.send(broadcast)
print(f"broadcast seq={sent.seq} -> arc.channel.project-alpha")

# Multi-target send: same envelope, multiple URIs.
multi = Message(
    sender="agent://planner",
    to=["channel://project-alpha", "agent://writer"],
    msg_type=MsgType.TASK,
    priority=Priority.HIGH,
    action_required=True,
    body="Draft the kickoff agenda; everyone heads-up via channel.",
    refs=["task://T-003"],
)
sent = await svc.send(multi)
print(f"multi-target seq={sent.seq} (last stream's seq)")

## 6. Receiving — cursor-driven `poll` and `ack`

Sending is half the story. `arcteam` is *pull-based* — the sender does not push to a callback. Each receiver maintains a `Cursor` per stream that records the last `seq` they processed. `poll` reads forward from that cursor without advancing it; `ack` advances the cursor only after the receiver has handled (or persisted) the work.

This separation is deliberate. It means crashes between *receipt* and *processing* are recoverable — restart, re-poll, and you'll see the same message again because the cursor never moved. It also means the receiver — not the service — controls flow. Slow receivers do not slow senders.

| Method | Effect |
|---|---|
| `poll(stream, entity_id, max_messages)` | Read messages with `seq > cursor.seq`, up to `max_messages`. Cursor unchanged. |
| `ack(stream, entity_id, seq, byte_pos)` | Move cursor forward to `seq`. Forward-only — backward acks are silently rejected. |
| `get_cursor(stream, entity_id)` | Inspect current cursor. Returns `None` before the first ack. |
| `poll_all(entity_id, max_per_stream)` | Convenience: poll inbox + every role stream + every channel the entity belongs to. |

In [ ]:
# Researcher's inbox — DM task from §5 should be sitting there.
inbox = await svc.poll("arc.agent.researcher", "agent://researcher")
print(f"researcher inbox has {len(inbox)} message(s):")
for m in inbox:
    payload = json.loads(m.body)
    print(
        f"  seq={m.seq}  type={m.msg_type.value}  priority={m.priority.value}  task={payload['task_id']}"
    )

Polling is non-destructive: read it again and the same message comes back. Only `ack` moves the cursor.

In [ ]:
# Re-poll without ack: same message returns.
again = await svc.poll("arc.agent.researcher", "agent://researcher")
print("re-poll without ack:", [m.seq for m in again])

# Process and ack.
for m in inbox:
    # ... receiver-specific work happens here ...
    await svc.ack("arc.agent.researcher", "agent://researcher", seq=m.seq, byte_pos=0)

# Now the inbox is empty for this consumer.
drained = await svc.poll("arc.agent.researcher", "agent://researcher")
print("after ack:", [m.seq for m in drained])

cur = await svc.get_cursor("arc.agent.researcher", "agent://researcher")
print(f"cursor: stream={cur.stream}  seq={cur.seq}  consumer={cur.consumer}")

**Cursors are forward-only.** A buggy or malicious receiver cannot "rewind" past acked work. If you ack to `seq=5` then ack `seq=2`, the second ack is silently dropped (with a `WARNING` log).

In [ ]:
import logging

logging.getLogger("arcteam.messenger").setLevel(logging.ERROR)  # quiet the warning for clarity

# Send a few channel messages, ack to seq=3, try to rewind to seq=1.
for i in range(3):
    await svc.send(
        Message(
            sender="agent://planner",
            to=["channel://project-alpha"],
            body=f"status update {i}",
        )
    )
before = await svc.get_cursor("arc.channel.project-alpha", "agent://researcher")
before_seq = before.seq if before else 0

msgs = await svc.poll("arc.channel.project-alpha", "agent://researcher")
if msgs:
    target_seq = msgs[-1].seq
    await svc.ack("arc.channel.project-alpha", "agent://researcher", seq=target_seq, byte_pos=0)
    # Backward attempt — silently rejected.
    await svc.ack("arc.channel.project-alpha", "agent://researcher", seq=before_seq, byte_pos=0)

after = await svc.get_cursor("arc.channel.project-alpha", "agent://researcher")
print(f"final cursor seq: {after.seq}  (cannot go backward)")

**`poll_all` — drain every subscribed stream in one call.** Each agent has an inbox plus role streams plus channel streams; in real loops you want to fan in across all of them in one round trip.

In [ ]:
all_streams = await svc.poll_all("agent://writer", max_per_stream=10)
print("writer poll_all results:")
for stream, msgs in all_streams.items():
    print(f"  {stream}: {len(msgs)} message(s)")
    for m in msgs:
        body_preview = m.body[:60] + ("..." if len(m.body) > 60 else "")
        print(f"    seq={m.seq}  type={m.msg_type.value}  body={body_preview!r}")

## 7. Priority-based ordering

Streams are FIFO by `seq`. The storage backend does *not* reorder messages by priority — that's a deliberate choice for predictability and audit replay. Two messages sent at times *T0* and *T1* with `T0 < T1` will always have `seq(T0) < seq(T1)` regardless of priority.

**Priority is a receiver-side hint.** The convention is: poll the stream, sort the in-memory batch by priority, drain `CRITICAL → HIGH → NORMAL → LOW`. This keeps the audit chain linear (the order messages *arrived* is the order in storage) while still letting receivers honor urgency. We codify the priority comparator next.

In [ ]:
# Send a mixed-priority batch to the writer's inbox in a deliberately wrong order.
for body, prio, mtype in [
    ("low-priority background hint", Priority.LOW, MsgType.INFO),
    ("normal task", Priority.NORMAL, MsgType.TASK),
    ("page now! incident in prod", Priority.CRITICAL, MsgType.ALERT),
    ("urgent followup", Priority.HIGH, MsgType.REQUEST),
    ("fyi note", Priority.LOW, MsgType.INFO),
]:
    await svc.send(
        Message(
            sender="agent://planner",
            to=["agent://writer"],
            msg_type=mtype,
            priority=prio,
            body=body,
        )
    )

raw = await svc.poll("arc.agent.writer", "agent://writer")
print("raw FIFO order (by seq):")
for m in raw:
    print(
        f"  seq={m.seq}  prio={m.priority.value:<8}  type={m.msg_type.value:<8}  body={m.body!r}"
    )

Now sort that same batch using the `PRIORITY_RANK` mapping from §3. The receiver decides drain order; the audit chain still reflects send order.

In [ ]:
def drain_order(messages: list[Message]) -> list[Message]:
    """Stable sort by priority rank. CRITICAL first, LOW last; ties keep send order."""
    return sorted(messages, key=lambda m: (PRIORITY_RANK[m.priority], m.seq))


print("receiver-side drain order:")
for m in drain_order(raw):
    print(f"  prio={m.priority.value:<8}  seq={m.seq}  body={m.body!r}")

**Why not let the storage backend reorder?** Three reasons:

1. *Audit linearity.* A reviewer reading the stream wants "this is what happened in this order", not "this is what was processed in some priority order". Mixing the two breaks investigation.
2. *Predictable replay.* Cursors are forward-only. If priority could shuffle entries, an old cursor would mean different messages on different replays.
3. *Composition.* Different consumers of the same stream may want different drain policies. Letting each receiver sort in-memory is strictly more flexible than baking one policy into the backend.

If you need *true* priority queues with strict ordering guarantees, run separate streams per priority (e.g. `agent://writer-critical`, `agent://writer-normal`) and have the receiver poll them in priority order.

In [ ]:
# Drain in priority order, ack each as we go.
for m in drain_order(raw):
    await svc.ack("arc.agent.writer", "agent://writer", seq=m.seq, byte_pos=0)
    print(f"  acked seq={m.seq}  prio={m.priority.value}")

remaining = await svc.poll("arc.agent.writer", "agent://writer")
print(f"\nwriter inbox after drain: {len(remaining)} messages remain")

## 8. Failure and retry semantics

`MessagingService.send` validates four conditions on every send. Each failed condition routes the offending envelope to the **Dead Letter Queue (DLQ)** — a separate stream that captures the original message *plus* a `dlq_reason` and `dlq_timestamp` in `meta`. The exception is also re-raised so the sender knows the call failed.

| Condition | DLQ reason | Where checked |
|---|---|---|
| Sender not in registry | `sender_unauthorized` | First — before any routing |
| `body` exceeds `MAX_BODY_BYTES` (64 KB) | `body_too_large` | Before each routing pass |
| `to:` URI is malformed (wrong scheme, bad chars) | `invalid_address` | Per-URI |
| Sender is not a member of `channel://X` | `not_channel_member` | Per-URI for channel targets |

DLQ entries are fully introspectable via `dlq_list()`. The original `Message` round-trips intact — you can rebuild it, fix the offending field, and resend. There is no automatic retry; that's a deliberate "fail loud, let humans decide" posture.

In [ ]:
# 8.1 Unregistered sender.
try:
    await svc.send(
        Message(
            sender="agent://ghost",
            to=["channel://project-alpha"],
            body="who am I?",
        )
    )
except ValueError as e:
    print("sender_unauthorized -> ValueError:", e)

In [ ]:
# 8.2 Invalid URI scheme.
try:
    await svc.send(
        Message(
            sender="agent://planner",
            to=["http://not-a-real-arc-uri"],
            body="test",
        )
    )
except ValueError as e:
    print("invalid_address -> ValueError:", e)

In [ ]:
# 8.3 Channel membership: writer is a member of project-alpha; outsider is not.
await svc.create_channel(Channel(name="private-room", members=["agent://planner"]))
try:
    await svc.send(
        Message(
            sender="agent://writer",
            to=["channel://private-room"],
            body="smuggling in",
        )
    )
except ValueError as e:
    print("not_channel_member -> ValueError:", e)

In [ ]:
# 8.4 Body too large via Pydantic-bypass (model_construct).
# Pydantic catches this at construction; model_construct skips validation,
# so the service-level check fires and the DLQ records body_too_large.
from arcteam.types import MAX_BODY_BYTES

big_body = "x" * (MAX_BODY_BYTES + 100)
oversized = Message.model_construct(
    seq=0,
    id="",
    ts="",
    sender="agent://planner",
    to=["channel://project-alpha"],
    thread_id=None,
    msg_type=MsgType.INFO,
    priority=Priority.NORMAL,
    action_required=False,
    body=big_body,
    refs=[],
    status="sent",
    meta={},
)
try:
    await svc.send(oversized)
except ValueError as e:
    print("body_too_large -> ValueError:", e)

Inspect the DLQ. Every failed send above should be there with its reason.

In [ ]:
dlq = await svc.dlq_list(limit=50)
print(f"DLQ has {len(dlq)} entries:")
for entry in dlq:
    reason = entry["meta"].get("dlq_reason")
    sender = entry["sender"]
    targets = entry["to"]
    print(f"  reason={reason:<22}  sender={sender}  to={targets}")

### Manual retry pattern

There's no automatic retry by design — the four failure modes above are *configuration* errors, not transient ones. The right response is to fix the cause and resend. Below: a tiny helper that pulls a DLQ entry and rebuilds the `Message` for a corrected resend.

In [ ]:
def rebuild_from_dlq(entry: dict) -> Message:
    """Strip auto-assigned and DLQ metadata, return a fresh Message ready to send."""
    clean = {k: v for k, v in entry.items() if k not in {"seq", "id", "ts", "thread_id"}}
    clean["meta"] = {k: v for k, v in clean.get("meta", {}).items() if not k.startswith("dlq_")}
    return Message(**clean)


# Fix-and-resend the channel-membership offender by joining first.
membership_failure = next(d for d in dlq if d["meta"]["dlq_reason"] == "not_channel_member")
rebuilt = rebuild_from_dlq(membership_failure)

# Add the writer to the private-room channel so the resend succeeds.
await svc.join_channel("private-room", "agent://writer")
resent = await svc.send(rebuilt)
print(f"resent successfully: id={resent.id}  seq={resent.seq}")

### Audit residue

Every successful send leaves a `message.sent` record in the audit chain (one per routed URI). Failed sends *don't* — the rejection is captured in the DLQ instead, and exceptions are re-raised. This is intentional: the audit chain is the immutable record of *what was delivered*; the DLQ is the salvage pile for *what was rejected*.

Verify the audit chain end-to-end and inspect the message-sent footprint.

In [ ]:
ok, last_seq = await audit.verify_chain()
print(f"chain valid? {ok}  (verified through seq={last_seq})")
assert ok

records = await backend.read_stream("audit", "audit", after_seq=0, limit=10_000)
by_event: dict[str, int] = {}
for r in records:
    et = r["event_type"]
    by_event[et] = by_event.get(et, 0) + 1

print(f"\naudit summary across {len(records)} records:")
for et, count in sorted(by_event.items()):
    print(f"  {et:<24} {count}")

## 9. Summary

**What you saw in this notebook:**

- Why task distribution belongs to `arcteam` — not `arcllm`, not `arcrun`, not `arcagent`. The boundary is enforced by what each layer is allowed to know about.
- The full surface of `MsgType` (`INFO`, `REQUEST`, `TASK`, `RESULT`, `ALERT`, `ACK`) and `Priority` (`LOW`, `NORMAL`, `HIGH`, `CRITICAL`) — both are `StrEnum`, both are typed metadata for receivers to interpret.
- `Channel` — a member-gated shared topic with FR-7 enforcement (non-members rejected at send time).
- The 13-field `Message` envelope — auto-assigned `seq`/`id`/`ts`/`thread_id`, manual `sender`/`to`/`body`, optional metadata. 64 KB body cap enforced at construction.
- Three task-distribution patterns from one envelope: DM (`agent://`), role fan-out (`role://`), channel broadcast (`channel://`), plus multi-target sends.
- The pull-based receive loop: `poll` reads forward, `ack` advances cursor, cursors are forward-only, `poll_all` fans in across every subscribed stream.
- Priority is a *receiver-side* hint — streams are FIFO by `seq` for audit linearity. The receiver sorts and drains; the storage layer does not.
- Four send-time failure modes (`sender_unauthorized`, `body_too_large`, `invalid_address`, `not_channel_member`), all routed to the DLQ with `dlq_reason` and `dlq_timestamp` plus the original envelope. No automatic retry; manual rebuild-and-resend pattern shown.
- Audit chain holds end-to-end through every successful send + channel mutation; DLQ is the salvage record for rejections.

**Public API surface covered:**

- `arcteam.MsgType` — all six values
- `arcteam.Priority` — all four values
- `arcteam.Channel` — record shape, `create_channel`, `join_channel`, `list_channels`
- `arcteam.Message` — full field set, `MAX_BODY_BYTES`, Pydantic validation, `model_construct` for tests
- `arcteam.MessagingService` — `send`, `poll`, `poll_all`, `ack`, `get_cursor`, `dlq_list`
- `arcteam.Cursor` — read position primitive
- DID binding via `arctrust.AgentIdentity` for sender/receiver registry handles

**Where this fits in the four pillars:**

- **Identity** — every sender's URI handle is bound to a real DID via `arctrust.AgentIdentity`; unregistered senders are rejected.
- **Sign** — message signing/verification (covered next in `03-messaging-channels`) reuses `arctrust.sign`/`verify`; this notebook stays at the routing layer.
- **Authorize** — channel membership is enforced (FR-7). Cross-team policy decisions still go through `arctrust.policy.PolicyPipeline`.
- **Audit** — every routed message lands in the HMAC-chained audit stream; rejections land in the DLQ.

**Next:**

- `arcteam/03-messaging-channels.ipynb` — `MessagingService` in depth: signed envelopes, threading, channel lifecycle, consensus patterns.
- `arcteam/04-team-persistence.ipynb` — `TeamMemoryService`, `FileBackend`, durability after restart, recovery from a crashed team.